#**💰 Cash Flow Forecasting Model**
UMDAC Datathon Project (Group: WOW)

This notebook demonstrates the process of forecasting weekly cash flow using historical transaction data.

We apply simple time-series methods and evaluate their performance.

In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta

from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

In [ ]:
df = pd.read_csv("datarealll.csv")
df["Week_Start"] = pd.to_datetime(df["Week_Start"])

df.head()

## 🔄 Data Preparation

We aggregate transaction data into weekly cash flow and compute net cash flow.

In [ ]:
weekly = (
    df.groupby("Week_Start", as_index=False)
    .agg(
        Cash_In=("Cash_In_USD", "sum"),
        Cash_Out=("Cash_Out_USD", "sum")
    )
    .sort_values("Week_Start")
)

weekly["Net_Cash_Flow"] = weekly["Cash_In"] - weekly["Cash_Out"]

weekly.head()

In [ ]:
weekly.set_index("Week_Start")["Net_Cash_Flow"].plot(title="Weekly Net Cash Flow")

In [ ]:
def moving_average_forecast(series, window=4, steps=8):
    values = list(series)
    forecasts = []

    for _ in range(steps):
        next_val = np.mean(values[-window:])
        forecasts.append(next_val)
        values.append(next_val)

    return forecasts

In [ ]:
BACKTEST_WEEKS = 8

train = weekly.iloc[:-BACKTEST_WEEKS]
test = weekly.iloc[-BACKTEST_WEEKS:]

In [ ]:
forecast_values = moving_average_forecast(
    train["Net_Cash_Flow"].values,
    window=4,
    steps=BACKTEST_WEEKS
)

forecast_values

## 📏 Model Evaluation

We evaluate model performance using MAPE and MAE.


In [ ]:
actual = test["Net_Cash_Flow"].values
forecast = np.array(forecast_values)

mae = mean_absolute_error(actual, forecast)
mape = mean_absolute_percentage_error(actual, forecast) * 100

print("MAE:", mae)
print("MAPE:", mape)

In [ ]:
FUTURE_WEEKS = 4

future_forecast = moving_average_forecast(
    weekly["Net_Cash_Flow"].values,
    window=4,
    steps=FUTURE_WEEKS
)

last_week = weekly["Week_Start"].max()

future_dates = [
    last_week + timedelta(weeks=i+1)
    for i in range(FUTURE_WEEKS)
]

future_df = pd.DataFrame({
    "Week_Start": future_dates,
    "Forecast": future_forecast
})

future_df



## 🧠 Conclusion

- The model shows limited forecasting accuracy due to high volatility in cash flow data  
- Simple models such as moving average struggle to capture irregular financial patterns  
- Forecast results should be interpreted cautiously  



### Key Learning:
This project highlights the importance of:
- Evaluating model performance  
- Understanding limitations of simple models  
- Using forecasts as guidance, not absolute truth  